<a href="https://colab.research.google.com/github/KinzaAsif2456/discoverey/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [12]:
import os, sys, subprocess
import pandas as pd
import numpy as np

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Starter data found. You're ready.


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**RULE:**

> Add blockquote


"I want to pull pages where performance is dropping off compared to their overall search visibility. Basically, I'm flagging content that's either getting old or ranking in the Top 20 but failing to turn those impressions into actual clicks. I also set up a high-priority "High Stale Decoy" bucket—pages that haven't been touched in forever but still draw massive search volume. Those are our highest-risk/highest-reward updates."

**Reason Codes:**

HIGH_STALE_DECOY: Older 6 months old (181+) AND top-tier impressions (>5,000 90d).

CTR_UNDERPERFORM: Sitting in the Top 20 with a CTR under 0.5%.

STALE_CONTENT: Content hasn't been updated in over 90 days.

COMBO_STALE_CTR: Triggered both the staleness and low-CTR flags.

**Why I chose these thresholds:**


- I used 90 days because content that has not been updated for more than three months is more likely to become outdated.
- I used a CTR threshold of 0.5% because pages ranking in the top 20 should generally receive more clicks. A lower CTR suggests that the title, meta description, or content may need improvement.
- I also required at least 500 impressions so that very low-traffic pages do not trigger the rule based on noisy data.

In [13]:
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
base_rate = df["is_declining_label"].mean()
print(f"rows: {len(df)}, base decline rate: {base_rate:.3f}")

rows: 30000, base decline rate: 0.542


In [14]:
# Signal 1: staleness (behind FlyRank's refresh flags) ---
order = ["0-30", "31-90", "91-180", "181+"]
staleness_table = (df.groupby("freshness_tier")
                      .agg(n=("is_declining_label", "size"),
                           decline_rate=("is_declining_label", "mean"))
                      .reindex(order))
print(staleness_table)

                    n  decline_rate
freshness_tier                     
0-30            20480      0.511377
31-90             175      0.588571
91-180           9171      0.611057
181+              174      0.471264


VERDICT: MIXED

Decline rate rises steadily from 0–30 days up through 91–180 days. However, it drops in the 181+ days bucket. Because $n=174$ in the 181+ tier (compared to $n>20,000$ in newer tiers), this drop is likely just small-sample noise rather than a true drop in decay rate.

In [15]:
# Signal 2: CTR-vs-position (behind the CTR-fix logic) ---
df["low_ctr_flag"] = (
    (df["impressions_90d"] >= 500) &
    (df["avg_position"] > 0) & (df["avg_position"] <= 20) &
    (df["ctr"] < 0.005)
)

ctr_table = (df.groupby("low_ctr_flag")
               .agg(n=("is_declining_label", "size"),
                    decline_rate=("is_declining_label", "mean")))
print(ctr_table)



                  n  decline_rate
low_ctr_flag                     
False         28784      0.532310
True           1216      0.773026


VERDICT: CONFIRMED

Pages with low CTR in the top 20 consistently show a higher decline rate than average, confirming CTR performance as a solid signal.

I didn't want pages with only a few impressions to dominate the results. A CTR based on 20 or 30 impressions can fluctuate a lot, so I required at least 500 impressions before applying the CTR rule.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [16]:

# Thresholds based on Signal Audit
STALE_TIERS = ["91-180", "181+"]
CTR_THRESHOLD = 0.005 # 0.5%
IMPRESSION_DECOY_LEVEL = 5000

# 1. Flag components
df["stale_flag"] = df["freshness_tier"].isin(STALE_TIERS)
df["low_ctr_flag"] = (df["avg_position"] <= 20) & (df["ctr"] < CTR_THRESHOLD) & (df["impressions_90d"] >= 500)
df["is_decoy"] = (df["freshness_tier"] == "181+") & (df["impressions_90d"] >= IMPRESSION_DECOY_LEVEL)

# 2. Score logic (Weights: Decoy = 3, Combo = 2, Single = 1)
def calculate_score(row):
    if row["is_decoy"]: return 3
    if row["stale_flag"] and row["low_ctr_flag"]: return 2
    if row["stale_flag"] or row["low_ctr_flag"]: return 1
    return 0

df["score"] = df.apply(calculate_score, axis=1)

# 3. Reason code implementation (Ensures alignment with Section 1)
def get_reason(row):
    if row["is_decoy"]: return "HIGH_STALE_DECOY"
    if row["stale_flag"] and row["low_ctr_flag"]: return "COMBO_STALE_CTR"
    if row["low_ctr_flag"]: return "CTR_UNDERPERFORM"
    if row["stale_flag"]: return "STALE_CONTENT"
    return "NO_SIGNAL"

df["reason_code"] = df.apply(get_reason, axis=1)

# 4. Action labels
df["action"] = df["score"].apply(lambda s: "REFRESH_NOW" if s >= 2 else ("REVIEW" if s == 1 else "NO_ACTION"))

# Rank by score first, then impressions to prioritize 'big' pages
queue = df.sort_values(["score", "impressions_90d"], ascending=[False, False]).reset_index(drop=True)
queue["rank"] = queue.index + 1

# Export
os.makedirs("work/outputs", exist_ok=True)
out_cols = ["content_id", "client_id", "rank", "score", "reason_code", "action", "is_declining_label"]
queue[out_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"Queue Size: {len(queue)} | Baseline Decline Rate: {df['is_declining_label'].mean():.2%}")

Queue Size: 30000 | Baseline Decline Rate: 54.21%


In [17]:
!head work/outputs/baseline_action_score.csv

content_id,client_id,rank,score,reason_code,action,is_declining_label
content_cf56e2e2e282,client_7f2253d7e2,1,3,HIGH_STALE_DECOY,REFRESH_NOW,1
content_7368877ea310,client_7f2253d7e2,2,3,HIGH_STALE_DECOY,REFRESH_NOW,1
content_1bfaa38ff26c,client_7f2253d7e2,3,3,HIGH_STALE_DECOY,REFRESH_NOW,1
content_0a91db491d14,client_7f2253d7e2,4,3,HIGH_STALE_DECOY,REFRESH_NOW,1
content_5feee3994adb,client_7f2253d7e2,5,3,HIGH_STALE_DECOY,REFRESH_NOW,1
content_c2d929d83eaa,client_7f2253d7e2,6,3,HIGH_STALE_DECOY,REFRESH_NOW,1
content_c8e9d6ab9013,client_19581e27de,7,2,COMBO_STALE_CTR,REFRESH_NOW,1
content_825a9788af8d,client_4e07408562,8,2,COMBO_STALE_CTR,REFRESH_NOW,1
content_8ba781dafa55,client_8527a891e2,9,2,COMBO_STALE_CTR,REFRESH_NOW,1


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [18]:
# Code to display top 20 for review
review_cols = ["content_id", "rank", "score", "reason_code", "action",
               "freshness_tier", "avg_position", "ctr", "impressions_90d"]
print(queue[review_cols].head(20).to_string(index=False))

          content_id  rank  score      reason_code      action freshness_tier  avg_position  ctr  impressions_90d
content_cf56e2e2e282     1      3 HIGH_STALE_DECOY REFRESH_NOW           181+          19.7 0.15            61678
content_7368877ea310     2      3 HIGH_STALE_DECOY REFRESH_NOW           181+          24.8 0.13            59472
content_1bfaa38ff26c     3      3 HIGH_STALE_DECOY REFRESH_NOW           181+          22.2 0.23            25715
content_0a91db491d14     4      3 HIGH_STALE_DECOY REFRESH_NOW           181+          10.5 0.49            13299
content_5feee3994adb     5      3 HIGH_STALE_DECOY REFRESH_NOW           181+          39.0 0.01             7812
content_c2d929d83eaa     6      3 HIGH_STALE_DECOY REFRESH_NOW           181+          17.9 0.20             7558
content_c8e9d6ab9013     7      2  COMBO_STALE_CTR REFRESH_NOW         91-180           9.7 0.00           208678
content_825a9788af8d     8      2  COMBO_STALE_CTR REFRESH_NOW         91-180           

**Top-20 Review (Data-Bound SME Critique)**

| Rank | Content ID | Reason Code | Key Metrics | Why it got flagged | What would make this wrong? (Data-Bound Limits) |
|------|------------|-------------|-------------|--------------------|--------------------------------------------------|
|1| `cf56e2e2e282` | HIGH_STALE_DECOY | 61.6k Imps, 15% CTR, Pos 1.0 | Old + massive volume. | CTR is already 15% at Rank 1. Age alone triggered this flag, so updating the text might accomplish nothing. |
|2 | `7368877ea310` | HIGH_STALE_DECOY | 59.4k Imps, Pos 24.8, 181+ Tier | Page 3 position with huge volume. | Position 24.8 is deep on Page 3. Low CTR is driven by rank depth, not necessarily decaying content quality. |
| 3 | `1bfaa38ff26c` | HIGH_STALE_DECOY | 25.7k Imps, Pos 22.2, 181+ Tier | Page 3 average position with high volume. | Position >20 means CTR is naturally near zero. High impression volume here might just be broad query matching. |
| 4 | `0a91db491d14` | HIGH_STALE_DECOY | 13.2k Imps, Pos 10.5, 181+ Tier | Hovering on the Page 1/2 boundary. | Position 10.5 sits on the boundary. Small rank shifts bias the CTR signal heavily. |
| 5 | `5feee3994adb` | HIGH_STALE_DECOY | 7.8k Imps, Pos 39.0, 181+ Tier | Stale + deep rank. | Pos 39.0 — of course CTR is basically zero, that's just deep on Page 4. |
| 6 | `c2d929d83eaa` | HIGH_STALE_DECOY | 7.5k Imps, Pos 17.9, 181+ Tier | Stale content near the bottom of Page 2. | Without page metadata, I can't confirm if this is evergreen reference content that doesn't need frequent edits. |
| 7 | `c8e9d6ab9013` | COMBO_STALE_CTR | 208.6k Imps, 0.00 CTR, Pos 9.0 | 200k+ impressions, zero clicks — odd. | 208k impressions at Pos 9 with 0 clicks points to a data/logging glitch, not bad text. |
| 8 | `825a9788af8d` | COMBO_STALE_CTR | 16.7k Imps, 0.00 CTR, Pos 5.6 | Position 5.6 with zero click conversion. | Zero clicks over 16.7k impressions suggests a tracking issue or an unrecorded redirect. |
| 9 | `8ba781dafa55` | COMBO_STALE_CTR | 16.1k Imps, 0.00 CTR, Pos 9.0 | High Page 1 visibility with zero clicks. | Third page in a row with exact 0.00 CTR under high volume—likely another analytics tracking gap. |
| 10 | `d3aaf7d5f2fc` | COMBO_STALE_CTR | 7.7k Imps, Pos 8.3 | Page 1 placement failing CTR cutoff. | I can't see the title or topic, so I can't verify if this page simply satisfies intent on the SERP itself. |
| 11 | `1de025a8c508` | COMBO_STALE_CTR | 7.0k Imps, Pos 16.0 | Bottom of Page 2. | Not sure if this is genuine content decay or just position drag. I'd want to see rank history over time before letting anyone rewrite this. |
| 12| `e15ede72712d` | COMBO_STALE_CTR | 6.7k Imps, Pos 16.1 | Stale signal + high decoy potential. | Honestly, no clear theory from these numbers alone — I'd need actual page logs or URL context to understand why it's lagging. |
| 13 | `5195668f06db` | COMBO_STALE_CTR | 6.6k Imps, Pos 5.2 | Rank 5.2 but failing the 0.5% CTR cutoff. | Pos 5.2 should get clicks. If an edit happened recently that isn't logged in freshness_tier, the flag is stale. |
| 14 | `c65ee459f729` | COMBO_STALE_CTR | 6.5k Imps, Pos 17.4 | Page 2 average rank with low CTR. | Ranking at 17.4 explains the low CTR far better than the freshness tier does. |
| 15 | `9648b7053d6f` | COMBO_STALE_CTR | 6.5k Imps, Pos 9.3 | Edge of Top 10 with staleness flag. | CTR could be fine relative to its specific query, but without query-level data we cannot tell. |
| 16 | `9ee24f9f28c3` | COMBO_STALE_CTR | 5.7k Imps, Pos 8.3 | Position 8.3 with low engagement. | No strong theory from numbers alone—would need actual URL/page logs to see why clicks are low. |
| 17 | `28a161c4e8c5` | COMBO_STALE_CTR | 5.2k Imps, Pos 18.5 | High impressions near Page 2 boundary. | Position 18.5 means very few users see this link. Low CTR is expected here regardless of age. |
| 18 | `b331a2c7719d` | COMBO_STALE_CTR | 5.1k Imps, Pos 11.5 | Floating just outside the Top 10. | Position 11.5 is the bottleneck. Pushing position up matters more than rewriting stale text. |
| 9 | `7722f40b12c1` | COMBO_STALE_CTR | 5.0k Imps, Pos 8.3 | Barely passes impression threshold. | Flagged as a decoy, but honestly the 5,000 impression decoy cutoff is a bit arbitrary — could just as easily be 4,000. |
| 20 | `92ca79f8b232` | COMBO_STALE_CTR | 4.6k Imps, Pos 7.5 | Score 2 trigger at 4.6k impressions. | Borderline volume. It is flagged, but at 4.6k impressions, the return on rewriting it might be minimal. |

**Note on Ranks 7, 8, and 9:**
Rows 7, 8, and 9 all show an exact 0.00 CTR despite pulling 16k to 208k impressions on Page 1. This isn't three separate content problems—it's almost certainly a single broken tracking pixel/analytics bug.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [19]:
print("Columns used in scoring: freshness_tier, avg_position, impressions_90d, ctr")
print("trend_direction / trend_pct used only to build is_declining_label (the label) —",
      "never inside stale_flag, low_ctr_flag, score, reason_code, or action.")

borderline = queue[queue["score"] == 1].sort_values("impressions_90d").head(3)
print(borderline[review_cols])

Columns used in scoring: freshness_tier, avg_position, impressions_90d, ctr
trend_direction / trend_pct used only to build is_declining_label (the label) — never inside stale_flag, low_ctr_flag, score, reason_code, or action.
                 content_id   rank  score    reason_code  action  \
10084  content_901b40631379  10085      1  STALE_CONTENT  REVIEW   
10019  content_00b467976cbd  10020      1  STALE_CONTENT  REVIEW   
9976   content_1e51704e6f13   9977      1  STALE_CONTENT  REVIEW   

      freshness_tier  avg_position  ctr  impressions_90d  
10084         91-180          15.0  0.0                1  
10019         91-180          10.0  0.0                1  
9976          91-180           9.0  0.0                1  


In [23]:
# Section 4: Inspecting low-volume flagged rows (score == 1)
borderline_flagged = queue[queue["score"] == 1].sort_values("impressions_90d").head(3)
print(borderline_flagged[["content_id", "rank", "score", "reason_code", "impressions_90d", "avg_position", "ctr", "freshness_tier"]])

                 content_id   rank  score    reason_code  impressions_90d  \
10084  content_901b40631379  10085      1  STALE_CONTENT                1   
10019  content_00b467976cbd  10020      1  STALE_CONTENT                1   
9976   content_1e51704e6f13   9977      1  STALE_CONTENT                1   

       avg_position  ctr freshness_tier  
10084          15.0  0.0         91-180  
10019          10.0  0.0         91-180  
9976            9.0  0.0         91-180  


**WEAK PICKS & BORDERLINE ANALYSIS:**

Looking at our lowest-impression rows with `score == 1`, a clear flaw in the baseline rule emerges:

1. **Unfiltered Low-Volume Noise (1 Impression Flaws):**
   Content IDs `content_901b40631379`, `content_00b467976cbd`, and `content_1e51704e6f13` all triggered `STALE_CONTENT` despite having an `impressions_90d` value of **1**.
   While our CTR rule required at least 500 impressions, our staleness rule didn't set a volume floor. Flagging pages with 1 impression over 3 months is a waste of writing resources—these are essentially dead pages where a refresh has virtually zero expected ROI.

2. **Position Drag conflated with Decay:**
   `content_901b40631379` sits at `avg_position` 15.0 (Page 2). Even though it's in the `91-180` day freshness tier, its lack of traffic is driven by its rank depth, not necessarily content decay.



**LEAKAGE CHECK:**

I verified that `is_declining_label` and `trend_direction` are strictly omitted from `score`, `reason_code`, and `action`. The baseline relies strictly on `freshness_tier`, `avg_position`, `impressions_90d`, and `ctr`. No target or future trend data leaked into the baseline logic.

## Conclusion

This baseline relies on two simple, explainable signals: content freshness and CTR performance. Both were audited beforehand and confirmed as strong indicators of decay. The scoring logic is easy to interpret and completely leak-free. While it won't catch every declining page, it gives us a clear benchmark for our ML models to compete against in the next sprint.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.